In [ ]:
# Elian Henrique V16 - 00:17 30/09
# Base (Aquisição de Dados e UI)
# Chatbot (Funcional)

!pip install --quiet --upgrade \
    langchain \
    langgraph \
    langchain-core \
    langchain-google-genai \
    pandas \
    boto3 \
    botocore \
    google-generativeai \
    gradio

import os
import pandas as pd
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import requests
from io import StringIO
from datetime import date, timedelta
import re
import gradio as gr

# LangChain / LangGraph (API nova)
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import Tool
from langchain_core.exceptions import OutputParserException
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool, Tool
from langchain_core.prompts import PromptTemplate


# Substitua pela sua chave de API do Google
os.environ['GOOGLE_API_KEY'] = =
s3_client = boto3.client('s3', region_name='sa-east-1', config=Config(signature_version=UNSIGNED))
bucket_name = 'ons-aws-prod-opendata'
s3_file_structure_cache = {}

categorias_datasets = {
    "Carga de Energia (Demanda)": [
        {"nome": "Carga de Energia Diária", "fonte": "s3", "valor": "carga_energia_di", "estrategia_carga": "preload"},
        {"nome": "Carga de Energia Mensal", "fonte": "s3", "valor": "carga_energia_me", "estrategia_carga": "preload"},
        {"nome": "Curva de Carga Horária", "fonte": "s3", "valor": "curva-carga-ho", "estrategia_carga": "ondemand"},
        {"nome": "Carga Global de Roraima", "fonte": "s3", "valor": "cargaglobal_roraima", "estrategia_carga": "preload"},
        {"nome": "Interrupção de Carga", "fonte": "s3", "valor": "interrupcao_carga", "estrategia_carga": "preload"},
    ],
    "Geração de Energia (Oferta)": [
        {"nome": "Capacidade Instalada de Geração", "fonte": "s3", "valor": "capacidade-geracao", "estrategia_carga": "preload"},
        {"nome": "Disponibilidade de Usinas (Horária)", "fonte": "s3", "valor": "disponibilidade_usina_ho", "estrategia_carga": "ondemand"},
        {"nome": "Geração por Usina (Horária)", "fonte": "s3", "valor": "geracao_usina_2_ho", "estrategia_carga": "ondemand"},
        {"nome": "Geração Térmica por Despacho (Horária)", "fonte": "s3", "valor": "geracao_termica_despacho_2_ho", "estrategia_carga": "ondemand"},
        {"nome": "Fator de Capacidade (Diário)", "fonte": "s3", "valor": "fator_capacidade_2_di", "estrategia_carga": "ondemand"},
        {"nome": "Modalidade de Operação de Usina", "fonte": "s3", "valor": "modalidade_usina", "estrategia_carga": "preload"},
        {"nome": "Unidade Geradora em Operação como Comp. Síncrono", "fonte": "s3", "valor": "uge_opera_csi", "estrategia_carga": "preload"},
        {"nome": "Programação vs Previsão (Eólica/Solar)", "fonte": "s3", "valor": "programacao_x_previsao", "estrategia_carga": "ondemand"},
    ],
    "Hidrologia e Reservatórios": [
        {"nome": "Dados Hidrológicos de Reservatórios - Diária", "fonte": "s3", "valor": "dados_hidrologicos_di", "estrategia_carga": "ondemand"},
        {"nome": "Dados Hidrológicos de Reservatórios - Horária", "fonte": "s3", "valor": "dados_hidrologicos_ho", "estrategia_carga": "ondemand"},
        {"nome": "EAR por Bacia (Diário)", "fonte": "s3", "valor": "ear_bacia_di", "estrategia_carga": "ondemand"},
        {"nome": "EAR por REE (Diário)", "fonte": "s3", "valor": "ear_ree_di", "estrategia_carga": "preload"},
        {"nome": "EAR por Reservatório (Diário)", "fonte": "s3", "valor": "ear_reservatorio_di", "estrategia_carga": "ondemand"},
        {"nome": "EAR por Subsistema (Diário)", "fonte": "s3", "valor": "ear_subsistema_di", "estrategia_carga": "preload"},
        {"nome": "ENA por Bacia (Diário)", "fonte": "s3", "valor": "ena_bacia_di", "estrategia_carga": "ondemand"},
        {"nome": "ENA por REE (Diário)", "fonte": "s3", "valor": "ena_ree_di", "estrategia_carga": "preload"},
        {"nome": "ENA por Reservatório (Diário)", "fonte": "s3", "valor": "ena_reservatorio_di", "estrategia_carga": "ondemand"},
        {"nome": "ENA por Subsistema (Diário)", "fonte": "s3", "valor": "ena_subsistema_di", "estrategia_carga": "preload"},
        {"nome": "Energia Vertida Turbinável (Horária)", "fonte": "s3", "valor": "energia_vertida_turbinavel_ho", "estrategia_carga": "ondemand"},
        {"nome": "Grandezas Fluviométricas (Diário)", "fonte": "s3", "valor": "grandezas_fluviometricas_di", "estrategia_carga": "ondemand"},
        {"nome": "Reservatórios", "fonte": "s3", "valor": "reservatorio", "estrategia_carga": "preload"},
        {"nome": "Volume de Espera Recomendado", "fonte": "s3", "valor": "res_volumeespera", "estrategia_carga": "ondemand"},
    ],
    "Operação e Fluxo de Energia": [
        {"nome": "Balanço DESSEM Detalhado", "fonte": "s3", "valor": "balanco_dessem_detalhe", "estrategia_carga": "ondemand"},
        {"nome": "Balanço DESSEM Geral", "fonte": "s3", "valor": "balanco_dessem_geral", "estrategia_carga": "ondemand"},
        {"nome": "Balanço Energia Subsistema (Horário)", "fonte": "s3", "valor": "balanco_energia_subsistema_ho", "estrategia_carga": "ondemand"},
        {"nome": "Intercâmbio Internacional (Horário)", "fonte": "s3", "valor": "intercambio_internacional_ho", "estrategia_carga": "ondemand"},
        {"nome": "Intercâmbio por Modalidade (Horário)", "fonte": "s3", "valor": "intercambio_modalidade_ho", "estrategia_carga": "ondemand"},
        {"nome": "Intercâmbio Nacional (Horário)", "fonte": "s3", "valor": "intercambio_nacional_ho", "estrategia_carga": "ondemand"},
        {"nome": "Programação Diária", "fonte": "s3", "valor": "programacao_diaria", "estrategia_carga": "ondemand"},
        {"nome": "Restrição de Operação Eólica", "fonte": "s3", "valor": "restricao_coff_eolica_tm", "estrategia_carga": "ondemand"},
        {"nome": "Restrição de Operação Eólica (Detalhada)", "fonte": "s3", "valor": "restricao_coff_eolica_detail_tm", "estrategia_carga": "ondemand"},
        {"nome": "Restrição de Operação Fotovoltaica", "fonte": "s3", "valor": "restricao_coff_fotovoltaica_tm", "estrategia_carga": "ondemand"},
        {"nome": "Restrição de Operação Fotovoltaica (Detalhada)", "fonte": "s3", "valor": "restricao_coff_fotovoltaica_detail_tm", "estrategia_carga": "ondemand"},
    ],
    "Dados Econômicos e Comerciais": [
        {"nome": "CMO Semanal", "fonte": "s3", "valor": "cmo_se", "estrategia_carga": "preload"},
        {"nome": "CMO Semi-Horário", "fonte": "s3", "valor": "cmo_tm", "estrategia_carga": "ondemand"},
        {"nome": "CVU das Usinas Térmicas", "fonte": "s3", "valor": "cvu_usitermica_se", "estrategia_carga": "preload"},
        {"nome": "Geração para Exportação Internacional", "fonte": "s3", "valor": "geracao_exportacao_internacional_ho", "estrategia_carga": "ondemand"},
        {"nome": "Importação Comercial", "fonte": "s3", "valor": "importacaoenergia-comercial-2-ho", "estrategia_carga": "ondemand"},
        {"nome": "Ofertas de Preço para Importação", "fonte": "s3", "valor": "ofertapreco-importacao-se", "estrategia_carga": "preload"},
    ],
    "Infraestrutura da Rede": [
        {"nome": "Capacidade de Transformação", "fonte": "s3", "valor": "capacidade-transformacao", "estrategia_carga": "preload"},
        {"nome": "Equipamentos de Controle de Reativos", "fonte": "s3", "valor": "equipamento_controle_reativo", "estrategia_carga": "preload"},
        {"nome": "Linhas de Transmissão", "fonte": "s3", "valor": "linha_transmissao", "estrategia_carga": "preload"},
        {"nome": "Subestação", "fonte": "s3", "valor": "subestacao", "estrategia_carga": "preload"},
    ],
    "Dados Relacionais (Mapeamento)": [
        {"nome": "Relacionamento Usina-Conjunto", "fonte": "s3", "valor": "usina_conjunto", "estrategia_carga": "preload"},
        {"nome": "Relacionamento Usina-Pequenas Usinas", "fonte": "s3", "valor": "usina_pqu", "estrategia_carga": "preload"},
    ],
}

descricoes_colunas = {
    # Carga de Energia (Demanda)
    "Carga de Energia Diária": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema',
        'din_instante': 'Data da medição', 'val_cargaenergiamwmed': 'Valor da Carga de Energia (MWméd)'
    },
    "Carga de Energia Mensal": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema',
        'din_instante': 'Mês da medição', 'val_cargaenergiamwmed': 'Valor da Carga de Energia (MWméd)'
    },
    "Curva de Carga Horária": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema',
        'din_instante': 'Data e Hora da medição', 'val_cargaenergiahomwmed': 'Valor da Carga de Energia Horária (MWméd)'
    },
    "Carga Global de Roraima": {
        'din_instante': 'Data e Hora da medição', 'val_carga_mw': 'Valor da Carga (MW)'
    },
    "Interrupção de Carga": {
        'cod_perturbacao': 'Código da Perturbação', 'din_interrupcaocarga': 'Data e Hora da Interrupção', 'id_subsistema': 'ID do Subsistema',
        'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_agente': 'Nome do Agente',
        'val_cargainterrompida_mw': 'Carga Interrompida (MW)', 'val_tempomedio_minutos': 'Tempo Médio de Interrupção (minutos)',
        'val_energianaosuprida_mwh': 'Energia Não Suprida (MWh)', 'flg_envolveuredebasica': 'Flag: Envolveu Rede Básica (S/N)',
        'flg_envolveuredeoperacao': 'Flag: Envolveu Rede de Operação (S/N)'
    },

    # Geração de Energia (Oferta)
    "Capacidade Instalada de Geração": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'nom_modalidadeoperacao': 'Modalidade de Operação', 'nom_agenteproprietario': 'Agente Proprietário', 'nom_agenteoperador': 'Agente Operador',
        'nom_tipousina': 'Tipo da Usina', 'nom_usina': 'Nome da Usina', 'ceg': 'Código Único do Empreendimento de Geração (CEG)',
        'nom_unidadegeradora': 'Nome da Unidade Geradora', 'cod_equipamento': 'Código do Equipamento', 'num_unidadegeradora': 'Número da Unidade Geradora',
        'nom_combustivel': 'Tipo de Combustível', 'dat_entradateste': 'Data de Entrada em Teste', 'dat_entradaoperacao': 'Data de Entrada em Operação',
        'dat_desativacao': 'Data de Desativação', 'val_potenciaefetiva': 'Potência Efetiva (MW)'
    },
    "Disponibilidade de Usinas (Horária)": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'nom_usina': 'Nome da Usina', 'id_tipousina': 'ID do Tipo de Usina', 'nom_tipocombustivel': 'Nome do Tipo de Combustível',
        'id_ons': 'ID ONS da Usina', 'ceg': 'CEG da Usina', 'din_instante': 'Data e Hora da Medição', 'val_potenciainstalada': 'Potência Instalada (MW)',
        'val_dispoperacional': 'Disponibilidade Operacional (MW)', 'val_dispsincronizada': 'Disponibilidade Sincronizada (MW)'
    },
    "Geração por Usina (Horária)": {
        'din_instante': 'Data e Hora da Medição', 'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado',
        'nom_estado': 'Nome do Estado', 'cod_modalidadeoperacao': 'Código da Modalidade de Operação', 'nom_tipousina': 'Tipo de Usina',
        'nom_tipocombustivel': 'Tipo de Combustível', 'nom_usina': 'Nome da Usina', 'id_ons': 'ID ONS da Usina', 'ceg': 'CEG da Usina',
        'val_geracao': 'Valor da Geração (MWméd)'
    },
    "Geração Térmica por Despacho (Horária)": {
        'din_instante': 'Data e Hora da Medição', 'nom_tipopatamar': 'Tipo de Patamar', 'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema',
        'nom_usina': 'Nome da Usina', 'cod_usinaplanejamento': 'Código da Usina (Planejamento)', 'ceg': 'CEG da Usina',
        'val_proggeracao': 'Programação da Geração (MWméd)', 'val_progordemmerito': 'Programação por Ordem de Mérito (MWméd)', 'val_progordemdemeritoref': 'val_progordemdemeritoref',
        'val_progordemdemeritoacimadainflex': 'val_progordemdemeritoacimadainflex', 'val_proginflexibilidade': 'Programação por Inflexibilidade (MWméd)',
        'val_proginflexembutmerito': 'val_proginflexembutmerito', 'val_proginflexpura': 'Programação por Inflexibilidade Pura (MWméd)',
        'val_prograzaoeletrica': 'Programação por Razão Elétrica (MWméd)', 'val_proggarantiaenergetica': 'Programação por Garantia Energética (MWméd)',
        'val_proggfom': 'Programação por Geração Fora da Ordem de Mérito (GFOM)', 'val_progreposicaoperdas': 'Programação para Reposição de Perdas (MWméd)',
        'val_progexportacao': 'Programação para Exportação (MWméd)', 'val_progreservapotencia': 'Programação para Reserva de Potência (MWméd)',
        'val_proggsub': 'Programação por Geração de Substituição (GSub)', 'val_progunitcommitment': 'Programação por Unit Commitment', 'val_progconstrainedoff': 'Programação por Constrained Off',
        'val_proginflexibilidadedessem': 'Programação por Inflexibilidade (DESSEM)', 'val_verifgeracao': 'Verificação da Geração (MWméd)', 'val_verifordemmerito': 'Verificação por Ordem de Mérito (MWméd)',
        'val_verifordemdemeritoacimadainflex': 'val_verifordemdemeritoacimadainflex', 'val_verifinflexibilidade': 'Verificação por Inflexibilidade (MWméd)', 'val_verifinflexembutmerito': 'val_verifinflexembutmerito',
        'val_verifinflexpura': 'Verificação por Inflexibilidade Pura (MWméd)', 'val_verifrazaoeletrica': 'Verificação por Razão Elétrica (MWméd)',
        'val_verifgarantiaenergetica': 'Verificação por Garantia Energética (MWméd)', 'val_verifgfom': 'Verificação por Geração Fora da Ordem de Mérito (GFOM)',
        'val_verifreposicaoperdas': 'Verificação para Reposição de Perdas (MWméd)', 'val_verifexportacao': 'Verificação para Exportação (MWméd)', 'val_fdexp': 'val_fdexp',
        'val_verifreservapotencia': 'Verificação para Reserva de Potência (MWméd)', 'val_atendsatisfatoriorpo': 'val_atendsatisfatoriorpo',
        'val_verifgsub': 'Verificação por Geração de Substituição (GSub)', 'val_verifunitcommitment': 'Verificação por Unit Commitment', 'val_verifconstrainedoff': 'Verificação por Constrained Off',
        'tip_restricaoeletrica': 'Tipo de Restrição Elétrica'
    },
    "Fator de Capacidade (Diário)": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'cod_pontoconexao': 'Código do Ponto de Conexão', 'nom_pontoconexao': 'Nome do Ponto de Conexão', 'nom_localizacao': 'Localização',
        'val_latitudesecoletora': 'Latitude da Subestação Coletora', 'val_longitudesecoletora': 'Longitude da Subestação Coletora',
        'val_latitudepontoconexao': 'Latitude do Ponto de Conexão', 'val_longitudepontoconexao': 'Longitude do Ponto de Conexão',
        'nom_modalidadeoperacao': 'Modalidade de Operação', 'nom_tipousina': 'Tipo da Usina', 'nom_usina_conjunto': 'Nome da Usina ou Conjunto',
        'id_ons': 'ID ONS da Usina', 'ceg': 'CEG da Usina', 'din_instante': 'Data da Medição',
        'val_geracaoprogramada': 'Geração Programada (MWméd)', 'val_geracaoverificada': 'Geração Verificada (MWméd)',
        'val_capacidadeinstalada': 'Capacidade Instalada (MW)', 'val_fatorcapacidade': 'Fator de Capacidade (%)'
    },
    "Modalidade de Operação de Usina": {
        'nom_usina': 'Nome da Usina', 'ceg': 'CEG da Usina', 'nom_modalidadeoperacao': 'Modalidade de Operação',
        'val_potenciaautorizada': 'Potência Autorizada (MW)', 'sgl_centrooperacao': 'Sigla do Centro de Operação', 'nom_pontoconexao': 'Ponto de Conexão',
        'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado', 'sts_aneel': 'Status na ANEEL'
    },
    "Unidade Geradora em Operação como Comp. Síncrono": {
        'dat_verificada': 'Data da Verificação', 'hora_verificada': 'Hora da Verificação', 'tip_usina': 'Tipo da Usina', 'nom_usina': 'Nome da Usina',
        'nom_unidadegeradora': 'Nome da Unidade Geradora', 'cod_usinadpp': 'Código da Usina (DPP)', 'cod_ugedpp': 'Código da Unidade Geradora (DPP)',
        'cod_pontomedicao': 'Código do Ponto de Medição', 'flg_operasincrono': 'Flag: Opera como Síncrono (S/N)', 'dsc_comentario': 'Comentário'
    },
    "Programação vs Previsão (Eólica/Solar)": {
        'dat_programacao': 'Data da Programação', 'num_patamar': 'Número do Patamar de Carga', 'cod_usinapdp': 'Código da Usina (PDP)',
        'nom_usinapdp': 'Nome da Usina (PDP)', 'val_previsao': 'Valor Previsto (MW)', 'val_programado': 'Valor Programado (MW)'
    },

    # Hidrologia e Reservatórios
    "Dados Hidrológicos de Reservatórios - Diária": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'tip_reservatorio': 'Tipo do Reservatório',
        'nom_bacia': 'Nome da Bacia', 'nom_ree': 'Nome do REE (Reservatório Equivalente de Energia)', 'id_reservatorio': 'ID do Reservatório', 'nom_reservatorio': 'Nome do Reservatório',
        'num_ordemcs': 'Número de Ordem em Cascata', 'cod_usina': 'Código da Usina', 'din_instante': 'Data da Medição',
        'val_nivelmontante': 'Nível de Montante (m)', 'val_niveljusante': 'Nível de Jusante (m)', 'val_volumeutilcon': 'Volume Útil Contínuo (%)',
        'val_vazaoafluente': 'Vazão Afluente (m³/s)', 'val_vazaoturbinada': 'Vazão Turbinada (m³/s)', 'val_vazaovertida': 'Vazão Vertida (m³/s)',
        'val_vazaooutrasestruturas': 'Vazão por Outras Estruturas (m³/s)', 'val_vazaodefluente': 'Vazão Defluente (m³/s)', 'val_vazaotransferida': 'Vazão Transferida (m³/s)',
        'val_vazaonatural': 'Vazão Natural (m³/s)', 'val_vazaoartificial': 'Vazão Artificial (m³/s)', 'val_vazaoincremental': 'Vazão Incremental (m³/s)',
        'val_vazaoevaporacaoliquida': 'Vazão de Evaporação Líquida (m³/s)', 'val_vazaousoconsuntivo': 'Vazão para Uso Consuntivo (m³/s)', 'val_vazaoincrementalbruta': 'Vazão Incremental Bruta (m³/s)'
    },
    "Dados Hidrológicos de Reservatórios - Horária": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'tip_reservatorio': 'Tipo de Reservatório', 'nom_bacia': 'Nome da Bacia',
        'id_reservatorio': 'ID do Reservatório', 'nom_reservatorio': 'Nome do Reservatório', 'cod_usina': 'Código da Usina', 'din_instante': 'Data e Hora da Medição',
        'val_nivelmontante': 'Nível de Montante (m)', 'val_niveljusante': 'Nível de Jusante (m)', 'val_volumeutil': 'Volume Útil (%)',
        'val_vazaoafluente': 'Vazão Afluente (m³/s)', 'val_vazaodefluente': 'Vazão Defluente (m³/s)', 'val_vazaoturbinada': 'Vazão Turbinada (m³/s)',
        'val_vazaovertida': 'Vazão Vertida (m³/s)', 'val_vazaooutrasestruturas': 'Vazão por Outras Estruturas (m³/s)', 'val_vazaotransferida': 'Vazão Transferida (m³/s)',
        'val_vazaovertidanaoturbinavel': 'Vazão Vertida Não Turbinável (m³/s)'
    },
    "EAR por Bacia (Diário)": {
        'nomecurto': 'Nome Curto da Bacia', 'ear_data': 'Data', 'ear_max_bacia': 'EAR Máxima da Bacia (MWmês)',
        'ear_verif_bacia_mwmes': 'EAR Verificada na Bacia (MWmês)', 'ear_verif_bacia_percentual': 'EAR Verificada na Bacia (%)'
    },
    "EAR por REE (Diário)": {
        'nom_ree': 'Nome do REE', 'ear_data': 'Data', 'ear_max_ree': 'EAR Máxima do REE (MWmês)',
        'ear_verif_ree_mwmes': 'EAR Verificada no REE (MWmês)', 'ear_verif_ree_percentual': 'EAR Verificada no REE (%)'
    },
    "EAR por Reservatório (Diário)": {
        'nom_reservatorio': 'Nome do Reservatório', 'cod_resplanejamento': 'Código do Reservatório (Planejamento)', 'tip_reservatorio': 'Tipo de Reservatório', 'nom_bacia': 'Nome da Bacia', 'nom_ree': 'Nome do REE',
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_subsistema_jusante': 'ID do Subsistema a Jusante', 'nom_subsistema_jusante': 'Nome do Subsistema a Jusante',
        'ear_data': 'Data', 'ear_reservatorio_subsistema_proprio_mwmes': 'EAR do Reservatório no Próprio Subsistema (MWmês)', 'ear_reservatorio_subsistema_jusante_mwmes': 'EAR do Reservatório no Subsistema a Jusante (MWmês)',
        'earmax_reservatorio_subsistema_proprio_mwmes': 'EAR Máxima do Reservatório no Próprio Subsistema (MWmês)', 'earmax_reservatorio_subsistema_jusante_mwmes': 'EAR Máxima do Reservatório no Subsistema a Jusante (MWmês)',
        'ear_reservatorio_percentual': 'EAR do Reservatório (%)', 'ear_total_mwmes': 'EAR Total (MWmês)', 'ear_maxima_total_mwmes': 'EAR Máxima Total (MWmês)',
        'val_contribearbacia': 'Contribuição EAR para Bacia', 'val_contribearmaxbacia': 'Contribuição EAR Máxima para Bacia', 'val_contribearsubsistema': 'Contribuição EAR para Subsistema',
        'val_contribearmaxsubsistema': 'Contribuição EAR Máxima para Subsistema', 'val_contribearsubsistemajusante': 'Contribuição EAR para Subsistema a Jusante', 'val_contribearmaxsubsistemajusante': 'Contribuição EAR Máxima para Subsistema a Jusante',
        'val_contribearsin': 'Contribuição EAR para o SIN', 'val_contribearmaxsin': 'Contribuição EAR Máxima para o SIN'
    },
    "EAR por Subsistema (Diário)": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'ear_data': 'Data',
        'ear_max_subsistema': 'EAR Máxima do Subsistema (MWmês)', 'ear_verif_subsistema_mwmes': 'EAR Verificada no Subsistema (MWmês)',
        'ear_verif_subsistema_percentual': 'EAR Verificada no Subsistema (%)'
    },
    "ENA por Bacia (Diário)": {
        'nom_bacia': 'Nome da Bacia', 'ena_data': 'Data', 'ena_bruta_bacia_mwmed': 'ENA Bruta da Bacia (MWméd)',
        'ena_bruta_bacia_percentualmlt': 'ENA Bruta da Bacia (% MLT)', 'ena_armazenavel_bacia_mwmed': 'ENA Armazenável da Bacia (MWméd)',
        'ena_armazenavel_bacia_percentualmlt': 'ENA Armazenável da Bacia (% MLT)'
    },
    "ENA por REE (Diário)": {
        'nom_reservatorioee': 'Nome do Reservatório Equivalente de Energia (REE)', 'ena_data': 'Data', 'ena_bruta_ree_mwmed': 'ENA Bruta do REE (MWméd)',
        'ena_bruta_ree_percentualmlt': 'ENA Bruta do REE (% MLT)', 'ena_armazenavel_ree_mwmed': 'ENA Armazenável do REE (MWméd)',
        'ena_armazenavel_ree_percentualmlt': 'ENA Armazenável do REE (% MLT)'
    },
    "ENA por Reservatório (Diário)": {
        'nom_reservatorio': 'Nome do Reservatório', 'cod_resplanejamento': 'Código do Reservatório (Planejamento)', 'tip_reservatorio': 'Tipo de Reservatório', 'nom_bacia': 'Nome da Bacia', 'nom_ree': 'Nome do REE',
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'ena_data': 'Data',
        'ena_bruta_res_mwmed': 'ENA Bruta do Reservatório (MWméd)', 'ena_bruta_res_percentualmlt': 'ENA Bruta do Reservatório (% MLT)',
        'ena_armazenavel_res_mwmed': 'ENA Armazenável do Reservatório (MWméd)', 'ena_armazenavel_res_percentualmlt': 'ENA Armazenável do Reservatório (% MLT)',
        'ena_queda_bruta': 'ENA por Queda Bruta', 'mlt_ena': 'Média de Longo Termo (MLT) da ENA'
    },
    "ENA por Subsistema (Diário)": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'ena_data': 'Data',
        'ena_bruta_regiao_mwmed': 'ENA Bruta da Região (MWméd)', 'ena_bruta_regiao_percentualmlt': 'ENA Bruta da Região (% MLT)',
        'ena_armazenavel_regiao_mwmed': 'ENA Armazenável da Região (MWméd)', 'ena_armazenavel_regiao_percentualmlt': 'ENA Armazenável da Região (% MLT)'
    },
    "Energia Vertida Turbinável (Horária)": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'nom_bacia': 'Nome da Bacia', 'nom_rio': 'Nome do Rio',
        'nom_agente': 'Nome do Agente', 'nom_reservatorio': 'Nome do Reservatório', 'cod_usina': 'Código da Usina', 'din_instante': 'Data e Hora da Medição',
        'val_geracao': 'Geração (MWméd)', 'val_disponibilidade': 'Disponibilidade (MW)', 'val_vazaoturbinada': 'Vazão Turbinada (m³/s)',
        'val_vazaovertida': 'Vazão Vertida (m³/s)', 'val_vazaovertidanaoturbinavel': 'Vazão Vertida Não Turbinável (m³/s)', 'val_produtividade': 'Produtividade (MW/m³/s)',
        'val_folgadegeracao': 'Folga de Geração (MWméd)', 'val_energiavertida': 'Energia Vertida (MWh)', 'val_vazaovertidaturbinavel': 'Vazão Vertida Turbinável (m³/s)',
        'val_energiavertidaturbinavel': 'Energia Vertida Turbinável (MWh)'
    },
    "Grandezas Fluviométricas (Diário)": {
        'id_postofluv': 'ID do Posto Fluviométrico', 'nom_postofluviometrico': 'Nome do Posto Fluviométrico', 'val_latitude': 'Latitude', 'val_longitude': 'Longitude',
        'nom_rio': 'Nome do Rio', 'nom_bacia': 'Nome da Bacia', 'din_medicao': 'Data da Medição',
        'val_vazaomedia': 'Vazão Média (m³/s)', 'val_vazaomediaincr': 'Vazão Média Incremental (m³/s)'
    },
    "Reservatórios": {
        'nom_reservatorio': 'Nome do Reservatório', 'tip_reservatorio': 'Tipo de Reservatório', 'cod_resplanejamento': 'Código do Reservatório (Planejamento)', 'cod_posto': 'Código do Posto',
        'nom_usina': 'Nome da Usina', 'ceg': 'CEG da Usina', 'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema',
        'nom_bacia': 'Nome da Bacia', 'nom_rio': 'Nome do Rio', 'nom_ree': 'Nome do REE', 'dat_entrada': 'Data de Entrada em Operação',
        'val_cotamaxima': 'Cota Máxima (m)', 'val_cotaminima': 'Cota Mínima (m)', 'val_volmax': 'Volume Máximo (hm³)',
        'val_volmin': 'Volume Mínimo (hm³)', 'val_volutiltot': 'Volume Útil Total (hm³)', 'val_produtibilidadeespecifica': 'Produtibilidade Específica',
        'val_produtividade65volutil': 'Produtividade a 65% do Volume Útil', 'val_tipoperda': 'Tipo de Perda Hidráulica', 'val_perda': 'Valor da Perda Hidráulica',
        'val_latitude': 'Latitude', 'val_longitude': 'Longitude', 'id_reservatorio': 'ID do Reservatório'
    },
    "Volume de Espera Recomendado": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'tip_reservatorio': 'Tipo de Reservatório', 'nom_bacia': 'Nome da Bacia',
        'nom_ree': 'Nome do REE', 'id_reservatorio': 'ID do Reservatório', 'nom_reservatorio': 'Nome do Reservatório', 'num_ordemcs': 'Número de Ordem em Cascata',
        'cod_usina': 'Código da Usina', 'din_instante': 'Data', 'val_volumeespera': 'Volume de Espera Recomendado (%)'
    },

    # Operação e Fluxo de Energia
    "Balanço DESSEM Detalhado": {
        'din_programacaodia': 'Data da Programação', 'num_patamar': 'Número do Patamar', 'cod_subsistema': 'Código do Subsistema',
        'val_demanda': 'Demanda (MWméd)', 'val_ger_hidraulica': 'Geração Hidráulica (MWméd)', 'val_ger_pch': 'Geração PCH (MWméd)',
        'val_ger_termica': 'Geração Térmica (MWméd)', 'val_ger_pct': 'Geração PCT (MWméd)', 'val_ger_eolica': 'Geração Eólica (MWméd)',
        'val_ger_fotovoltaica': 'Geração Fotovoltaica (MWméd)', 'val_ger_mmgd': 'Geração MMGD (MWméd)', 'val_cons_elevatoria': 'Consumo de Bombeamento (MWméd)'
    },
    "Balanço DESSEM Geral": {
        'din_programacaodia': 'Data da Programação', 'num_patamar': 'Número do Patamar', 'cod_subsistema': 'Código do Subsistema',
        'val_demanda': 'Demanda (MWméd)', 'val_geracao_renovavel': 'Geração Renovável (MWméd)', 'val_geracao_hidraulica': 'Geração Hidráulica (MWméd)',
        'val_geracao_termica': 'Geração Térmica (MWméd)', 'val_cons_elevatoria': 'Consumo de Bombeamento (MWméd)'
    },
    "Balanço Energia Subsistema (Horário)": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'din_instante': 'Data e Hora da Medição',
        'val_gerhidraulica': 'Geração Hidráulica (MWméd)', 'val_gertermica': 'Geração Térmica (MWméd)', 'val_gereolica': 'Geração Eólica (MWméd)',
        'val_gersolar': 'Geração Solar (MWméd)', 'val_carga': 'Carga (MWméd)', 'val_intercambio': 'Intercâmbio (MWméd)'
    },
    "Intercâmbio Internacional (Horário)": {
        'din_instante': 'Data e Hora da Medição', 'nom_paisdestino': 'País de Destino', 'val_intercambiomwmed': 'Valor do Intercâmbio (MWméd)'
    },
    "Intercâmbio por Modalidade (Horário)": {
        'nom_conversora': 'Nome da Conversora', 'din_instante': 'Data e Hora da Medição', 'val_modalidadecontratual': 'Valor na Modalidade Contratual (MWméd)',
        'val_modalidadeemergencial': 'Valor na Modalidade Emergencial (MWméd)', 'val_modalidadeoportunidade': 'Valor na Modalidade Oportunidade (MWméd)',
        'val_modalidadeteste': 'Valor na Modalidade Teste (MWméd)', 'val_modalidadeexcepcional': 'Valor na Modalidade Excepcional (MWméd)'
    },
    "Intercâmbio Nacional (Horário)": {
        'din_instante': 'Data e Hora da Medição', 'id_subsistema_origem': 'ID Subsistema de Origem', 'nom_subsistema_origem': 'Nome Subsistema de Origem',
        'id_subsistema_destino': 'ID Subsistema de Destino', 'nom_subsistema_destino': 'Nome Subsistema de Destino', 'val_intercambiomwmed': 'Valor do Intercâmbio (MWméd)'
    },
    "Programação Diária": {
        'din_programacaodia': 'Data da Programação', 'num_patamar': 'Número do Patamar', 'cod_exibicaousina': 'Código de Exibição da Usina', 'nom_usina': 'Nome da Usina',
        'tip_geracao': 'Tipo de Geração', 'nom_modalidadeoperacao': 'Modalidade de Operação', 'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema',
        'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado', 'val_geracaoprogramada': 'Geração Programada (MW)', 'val_disponibilidade': 'Disponibilidade (MW)',
        'val_ordemmerito': 'Valor por Ordem de Mérito (MW)', 'val_inflexibilidade': 'Valor por Inflexibilidade (MW)', 'val_uc': 'val_uc',
        'val_razaoeletrica': 'Valor por Razão Elétrica (MW)', 'val_geracaoenergetica': 'Valor por Geração Energética (MW)', 'val_gesubgsub': 'val_gesubgsub',
        'val_exportacao': 'Valor de Exportação (MW)', 'val_reposicaoexportacao': 'Valor de Reposição de Exportação (MW)', 'val_faltacombustivel': 'Valor por Falta de Combustível (MW)'
    },
    "Restrição de Operação Eólica": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'nom_usina': 'Nome da Usina', 'id_ons': 'ID ONS da Usina', 'ceg': 'CEG da Usina', 'din_instante': 'Data e Hora da Medição',
        'val_geracao': 'Geração (MWméd)', 'val_geracaolimitada': 'Geração Limitada (MWméd)', 'val_disponibilidade': 'Disponibilidade (MWméd)',
        'val_geracaoreferencia': 'Geração de Referência (MWméd)', 'val_geracaoreferenciafinal': 'Geração de Referência Final (MWméd)',
        'cod_razaorestricao': 'Código da Razão da Restrição', 'cod_origemrestricao': 'Código da Origem da Restrição', 'dsc_restricao': 'Descrição da Restrição'
    },
    "Restrição de Operação Eólica (Detalhada)": {
        'id_subsistema': 'ID do Subsistema', 'id_estado': 'UF do Estado', 'nom_modalidadeoperacao': 'Modalidade de Operação',
        'nom_conjuntousina': 'Nome do Conjunto de Usinas', 'nom_usina': 'Nome da Usina', 'id_ons': 'ID ONS da Usina', 'ceg': 'CEG da Usina',
        'din_instante': 'Data e Hora da Medição', 'val_ventoverificado': 'Vento Verificado (m/s)', 'flg_dadoventoinvalido': 'Flag: Dado de Vento Inválido',
        'val_geracaoestimada': 'Geração Estimada (MWméd)', 'val_geracaoverificada': 'Geração Verificada (MWméd)'
    },
    "Restrição de Operação Fotovoltaica": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'nom_usina': 'Nome da Usina', 'id_ons': 'ID ONS da Usina', 'ceg': 'CEG da Usina', 'din_instante': 'Data e Hora da Medição',
        'val_geracao': 'Geração (MWméd)', 'val_geracaolimitada': 'Geração Limitada (MWméd)', 'val_disponibilidade': 'Disponibilidade (MWméd)',
        'val_geracaoreferencia': 'Geração de Referência (MWméd)', 'val_geracaoreferenciafinal': 'Geração de Referência Final (MWméd)',
        'cod_razaorestricao': 'Código da Razão da Restrição', 'cod_origemrestricao': 'Código da Origem da Restrição', 'dsc_restricao': 'Descrição da Restrição'
    },
    "Restrição de Operação Fotovoltaica (Detalhada)": {
        'id_subsistema': 'ID do Subsistema', 'id_estado': 'UF do Estado', 'nom_modalidadeoperacao': 'Modalidade de Operação', 'nom_conjuntousina': 'Nome do Conjunto de Usinas',
        'nom_usina': 'Nome da Usina', 'id_ons': 'ID ONS da Usina', 'ceg': 'CEG da Usina', 'din_instante': 'Data e Hora da Medição',
        'val_irradianciaverificado': 'Irradiância Verificada (W/m²)', 'flg_dadoirradianciainvalido': 'Flag: Dado de Irradiância Inválido',
        'val_geracaoestimada': 'Geração Estimada (MWméd)', 'val_geracaoverificada': 'Geração Verificada (MWméd)'
    },

    # Dados Econômicos e Comerciais
    "CMO Semanal": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'din_instante': 'Data (Início da Semana Operativa)',
        'val_cmomediasemanal': 'CMO Médio Semanal (R$/MWh)', 'val_cmoleve': 'CMO Carga Leve (R$/MWh)',
        'val_cmomedia': 'CMO Carga Média (R$/MWh)', 'val_cmopesada': 'CMO Carga Pesada (R$/MWh)'
    },
    "CMO Semi-Horário": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'din_instante': 'Data e Hora da Medição', 'val_cmo': 'Custo Marginal de Operação (R$/MWh)'
    },
    "CVU das Usinas Térmicas": {
        'dat_iniciosemana': 'Data de Início da Semana', 'dat_fimsemana': 'Data de Fim da Semana', 'ano_referencia': 'Ano de Referência', 'mes_referencia': 'Mês de Referência',
        'num_revisao': 'Número da Revisão', 'nom_semanaoperativa': 'Nome da Semana Operativa', 'cod_usinaplanejamento': 'Código da Usina (Planejamento)',
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'nom_usina': 'Nome da Usina', 'val_cvu': 'Custo Variável Unitário (CVU) (R$/MWh)'
    },
    "Geração para Exportação Internacional": {
        'din_instante': 'Data e Hora da Medição', 'val_gerexpevt_ar': 'Geração para Exportação (EVT) para Argentina (MWméd)',
        'val_gerexpevt_uy': 'Geração para Exportação (EVT) para Uruguai (MWméd)', 'val_gerexptermica': 'Geração para Exportação Térmica (MWméd)',
        'val_exportacao_ar': 'Exportação para Argentina (MWméd)', 'val_exportacao_uy': 'Exportação para Uruguai (MWméd)'
    },
    "Importação Comercial": {
        'nom_pais': 'Nome do País', 'nom_agente': 'Nome do Agente', 'nom_bloco': 'Nome do Bloco de Energia', 'din_instante': 'Data e Hora da Medição',
        'val_importacaoprogramada': 'Importação Programada (MWméd)', 'val_importacaodespachada': 'Importação Despachada (MWméd)',
        'val_importacaoverificada': 'Importação Verificada (MWméd)', 'val_preco': 'Preço (R$/MWh)'
    },
    "Ofertas de Preço para Importação": {
        'nom_pais': 'Nome do País', 'nom_agente': 'Nome do Agente', 'nom_bloco': 'Nome do Bloco de Energia',
        'dat_iniciovalidade': 'Data de Início da Validade', 'dat_fimvalidade': 'Data de Fim da Validade', 'val_preco': 'Preço (R$/MWh)'
    },

    # Infraestrutura da Rede
    "Capacidade de Transformação": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'nom_tipotransformador': 'Tipo de Transformador', 'nom_agenteproprietario': 'Agente Proprietário', 'nom_subestacao': 'Nome da Subestação',
        'nom_transformador': 'Nome do Transformador', 'cod_equipamento': 'Código do Equipamento', 'dat_entradaoperacao': 'Data de Entrada em Operação',
        'dat_desativacao': 'Data de Desativação', 'val_tensaoprimario_kv': 'Tensão no Primário (kV)', 'val_tensaosecundario_kv': 'Tensão no Secundário (kV)',
        'val_tensaoterciario_kv': 'Tensão no Terciário (kV)', 'val_potencianominal_mva': 'Potência Nominal (MVA)',
        'nom_tipoderede': 'Tipo de Rede', 'num_barra_primario': 'Número da Barra do Primário', 'num_barra_secundario': 'Número da Barra do Secundário'
    },
    "Equipamentos de Controle de Reativos": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'nom_subestacao': 'Nome da Subestação', 'nom_agente_proprietario': 'Agente Proprietário', 'nom_tipoderede': 'Tipo de Rede',
        'nom_tipoequipamento': 'Tipo de Equipamento', 'nom_equipamento': 'Nome do Equipamento', 'val_potreativanominal_mvar': 'Potência Reativa Nominal (MVAr)',
        'val_limiteinferior_mvar': 'Limite Inferior (MVAr)', 'val_limitesuperior_mvar': 'Limite Superior (MVAr)',
        'dat_entradaoperacao': 'Data de Entrada em Operação', 'dat_desativacao': 'Data de Desativação', 'cod_equipamento': 'Código do Equipamento'
    },
    "Linhas de Transmissão": {
        'id_subsistema_terminalde': 'ID Subsistema (Origem)', 'nom_subsistema_terminalde': 'Nome Subsistema (Origem)', 'id_subsistema_terminalpara': 'ID Subsistema (Destino)', 'nom_subsistema_terminalpara': 'Nome Subsistema (Destino)',
        'id_estado_terminalde': 'UF Estado (Origem)', 'nom_estado_de': 'Nome Estado (Origem)', 'id_estado_terminalpara': 'UF Estado (Destino)', 'nom_estado_para': 'Nome Estado (Destino)',
        'nom_subestacao_de': 'Subestação (Origem)', 'nom_subestacao_para': 'Subestação (Destino)', 'val_niveltensao_kv': 'Nível de Tensão (kV)',
        'nom_tipoderede': 'Tipo de Rede', 'nom_tipolinha': 'Tipo de Linha', 'nom_agenteproprietario': 'Agente Proprietário', 'nom_linhadetransmissao': 'Nome da Linha de Transmissão',
        'cod_equipamento': 'Código do Equipamento', 'dat_entradaoperacao': 'Data de Entrada em Operação', 'dat_desativacao': 'Data de Desativação', 'dat_prevista': 'Data Prevista de Entrada',
        'val_comprimento': 'Comprimento (km)', 'val_resistencia': 'Resistência (pu)', 'val_reatancia': 'Reatância (pu)', 'val_shunt': 'Susceptância Shunt (pu)',
        'val_capacoperlongasemlimit': 'Capacidade Operacional Longa Duração S/ Limite (MW)', 'val_capacoperlongacomlimit': 'Capacidade Operacional Longa Duração C/ Limite (MW)',
        'val_capacopercurtasemlimit': 'Capacidade Operacional Curta Duração S/ Limite (MW)', 'val_capacopercurtacomlimit': 'Capacidade Operacional Curta Duração C/ Limite (MW)',
        'val_capacidadeoperveraodialonga': 'Cap. Op. Verão Dia Longa (MW)', 'val_capacidadeoperveraonoitelonga': 'Cap. Op. Verão Noite Longa (MW)', 'val_capacoperinvernodialonga': 'Cap. Op. Inverno Dia Longa (MW)',
        'val_capacoperinvernonoitelonga': 'Cap. Op. Inverno Noite Longa (MW)', 'val_capacoperveradiacurta': 'Cap. Op. Verão Dia Curta (MW)', 'val_capacoperveraonoitecurta': 'Cap. Op. Verão Noite Curta (MW)',
        'val_capacoperinvernodiacurta': 'Cap. Op. Inverno Dia Curta (MW)', 'val_capacoperinvernonoitecurta': 'Cap. Op. Inverno Noite Curta (MW)',
        'num_barra_de': 'Número da Barra (Origem)', 'num_barra_para': 'Número da Barra (Destino)'
    },
    "Subestação": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'id_estado': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'nom_agente_principal': 'Agente Principal', 'id_subestacao': 'ID da Subestação', 'nom_subestacao': 'Nome da Subestação',
        'val_niveltensao': 'Nível de Tensão (kV)', 'id_estacao': 'ID da Estação', 'num_barra': 'Número da Barra', 'val_latitude': 'Latitude', 'val_longitude': 'Longitude'
    },

    # Dados Relacionais (Mapeamento)
    "Relacionamento Usina-Conjunto": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'estad_id': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'id_tipousina': 'ID Tipo de Usina', 'id_conjuntousina': 'ID do Conjunto de Usinas', 'id_ons_conjunto': 'ID ONS do Conjunto',
        'id_ons_usina': 'ID ONS da Usina', 'nom_conjunto': 'Nome do Conjunto', 'nom_usina': 'Nome da Usina', 'ceg': 'CEG da Usina',
        'dat_iniciorelacionamento': 'Data de Início do Relacionamento', 'dat_fimrelacionamento': 'Data de Fim do Relacionamento'
    },
    "Relacionamento Usina-Pequenas Usinas": {
        'id_subsistema': 'ID do Subsistema', 'nom_subsistema': 'Nome do Subsistema', 'estad_id': 'UF do Estado', 'nom_estado': 'Nome do Estado',
        'id_tipousina': 'ID Tipo de Usina', 'id_ons_pequenasusinas': 'ID ONS de Pequenas Usinas', 'id_ons_usina': 'ID ONS da Usina',
        'nom_pequenasusinas': 'Nome Agregado de Pequenas Usinas', 'nom_usina': 'Nome da Usina', 'ceg': 'CEG da Usina'
    }
}

def extrair_ano_mes_do_arquivo(key):
    base_name = os.path.basename(key)
    match = re.search(r'_(\d{4})_(\d{2})\.csv', base_name, re.IGNORECASE)
    if match: return int(match.group(1)), int(match.group(2))
    match = re.search(r'(\d{4})-(\d{2})', base_name, re.IGNORECASE)
    if match: return int(match.group(1)), int(match.group(2))
    match = re.search(r'_(\d{4})\.csv', base_name, re.IGNORECASE)
    if match: return int(match.group(1)), None
    return None, None

def mapear_arquivos_s3(prefix):
    global s3_file_structure_cache
    if prefix in s3_file_structure_cache: return s3_file_structure_cache[prefix]
    file_structure = {}
    print(f"-> Mapeando arquivos S3 para: {prefix}...")
    try:
        paginator = s3_client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                if key.lower().endswith('.csv'):
                    year, month = extrair_ano_mes_do_arquivo(key)
                    if year:
                        if year not in file_structure: file_structure[year] = {}
                        if month: file_structure[year][month] = key
                        elif 'anual' not in file_structure[year]: file_structure[year]['anual'] = key
                    else:
                        if 'atemporal' not in file_structure: file_structure['atemporal'] = []
                        file_structure['atemporal'].append(key)
        s3_file_structure_cache[prefix] = file_structure
        return file_structure
    except Exception as e:
        print(f"Erro ao escanear S3: {e}")
        return {}

def corrigir_encoding_duplo(texto):
    """Repara strings que foram salvas em UTF-8 mas lidas como latin-1."""
    if isinstance(texto, str):
        try:
            return texto.encode('latin-1').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            return texto
    return texto

def carregar_dataframe_s3(file_keys):
    """Carrega, limpa encoding, e CONVERTE TIPOS de dados automaticamente."""
    if isinstance(file_keys, str): file_keys = [file_keys]
    all_dfs = []
    print(f"--> Carregando, limpando e tipando {len(file_keys)} arquivo(s)...")
    for file_key in file_keys:
        temp_filename = f"temp_{os.path.basename(file_key)}"
        df = None
        try:
            s3_client.download_file(bucket_name, file_key, temp_filename)

            try:
                df = pd.read_csv(temp_filename, sep=';', decimal=',', encoding='utf-8', low_memory=False)
            except Exception:
                df = pd.read_csv(temp_filename, sep=',', decimal='.', encoding='utf-8', low_memory=False)

            if df is None or df.empty:
                 raise ValueError("Não foi possível ler o CSV.")

            colunas_de_texto = df.select_dtypes(include=['object']).columns
            for coluna in colunas_de_texto:
                df[coluna] = df[coluna].apply(corrigir_encoding_duplo)


            for col in df.columns:

                if 'dat_' in col or 'din_' in col or col.endswith('_data'):
                    df[col] = pd.to_datetime(df[col], errors='coerce')

                elif 'val_' in col or col.startswith('num_'):
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            all_dfs.append(df)
        except Exception as e:
            print(f"   - ❌ Erro ao processar '{os.path.basename(file_key)}': {e}")
        finally:
            if os.path.exists(temp_filename): os.remove(temp_filename)

    if not all_dfs: return None, "Nenhum dado foi carregado com sucesso."

    final_df = pd.concat(all_dfs, ignore_index=True)
    print(f"   - Tipagem final do dataframe concluída.")
    return final_df, "Sucesso"

print("\nIniciando mapeamento e pré-carregamento...")
for datasets in categorias_datasets.values():
    for ds_info in datasets:
        mapear_arquivos_s3(f"dataset/{ds_info['valor']}/")

print("\n--- Pré-carregando datasets selecionados ---")
dataframes_precarregados = []
for datasets in categorias_datasets.values():
    for ds_info in datasets:
        if ds_info.get('estrategia_carga') == 'preload':
            print(f"-> Processando '{ds_info['nome']}'...")
            prefixo = f"dataset/{ds_info['valor']}/"
            estrutura = s3_file_structure_cache.get(prefixo, {})
            arquivos = []
            if estrutura:
              for item in estrutura.values():
                if isinstance(item, dict):
                  arquivos.extend(item.values())
                elif isinstance(item, list):
                  arquivos.extend(item)

            arquivos_unicos = sorted(list(set(arquivos)))
            if arquivos_unicos:
                df, _ = carregar_dataframe_s3(arquivos_unicos)
                if df is not None and not df.empty:
                    df['nome_do_dataset'] = ds_info['nome']
                    dataframes_precarregados.append(df)

print("\nUnindo dataframes pré-carregados...")
dataframe_unificado_para_ia = pd.concat(dataframes_precarregados, ignore_index=True) if dataframes_precarregados else pd.DataFrame()
print(f"✅ Dataframe unificado pronto! Formato: {dataframe_unificado_para_ia.shape}")

agent = None

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0, convert_system_message_to_human=False)

def verificar_disponibilidade(nome_dataset: str) -> str:
    global s3_file_structure_cache
    print(f"\n[Ferramenta de Verificação] Acionada para: {nome_dataset}")
    dataset_info = next((ds for cat in categorias_datasets.values() for ds in cat if ds['nome'] == nome_dataset), None)
    if not dataset_info: return f"Dataset '{nome_dataset}' não encontrado."
    prefixo = f"dataset/{dataset_info['valor']}/"
    file_structure = s3_file_structure_cache.get(prefixo, {})
    if not file_structure: return f"Nenhuma estrutura de arquivos foi encontrada para '{nome_dataset}'."
    anos_disponiveis = sorted([k for k in file_structure.keys() if isinstance(k, int)])
    if not anos_disponiveis: return f"O dataset '{nome_dataset}' é atemporal ou não possui dados anuais."
    output_detalhado = []
    for ano in anos_disponiveis:
        meses = sorted([m for m in file_structure[ano].keys() if isinstance(m, int)])
        if meses: output_detalhado.append(f"- {ano}: meses {meses}")
        elif 'anual' in file_structure[ano]: output_detalhado.append(f"- {ano}: possui um arquivo anual completo.")
    if not output_detalhado: return f"Não foi possível detalhar os meses para os anos: {anos_disponiveis}."
    return f"Dados para '{nome_dataset}' estão disponíveis para:\n" + "\n".join(output_detalhado)

def extrator_de_dados_por_data(query: str) -> str:
    try:
        nome_dataset, inicio_str, fim_str = [x.strip() for x in query.split('|')]
        data_inicio = pd.to_datetime(inicio_str).date()
        data_fim = pd.to_datetime(fim_str).date()
    except (ValueError, IndexError): return "Entrada inválida. Use o formato 'Nome do Dataset | AAAA-MM-DD | AAAA-MM-DD'."
    print(f"\n[Ferramenta de Extração] Acionada para: {nome_dataset}, Período: {inicio_str} a {fim_str}")
    dataset_info = next((ds for cat in categorias_datasets.values() for ds in cat if ds['nome'] == nome_dataset), None)
    if not dataset_info: return f"Dataset '{nome_dataset}' não encontrado."
    prefixo = f"dataset/{dataset_info['valor']}/"
    file_structure = s3_file_structure_cache.get(prefixo, {})
    anos_no_periodo = range(data_inicio.year, data_fim.year + 1)
    arquivos_para_carregar = []
    for ano in anos_no_periodo:
        if ano in file_structure:
            dados_do_ano = file_structure[ano]
            arquivos_para_carregar.extend([v for k, v in dados_do_ano.items() if isinstance(k, int) or k == 'anual'])
    if not arquivos_para_carregar: return f"Nenhum arquivo de dados encontrado para o período de {inicio_str} a {fim_str}."
    arquivos_para_carregar = sorted(list(set(arquivos_para_carregar)))

    df_full, status = carregar_dataframe_s3(arquivos_para_carregar)

    if df_full is None: return f"Falha ao carregar dados: {status}"
    coluna_data = next((col for col in df_full.columns if 'dat_' in col or 'din_' in col), None)
    if not coluna_data: return "Não foi possível encontrar a coluna de data para filtrar os dados carregados."
    df_full[coluna_data] = pd.to_datetime(df_full[coluna_data], errors='coerce')
    df_filtrado = df_full[(df_full[coluna_data].dt.date >= data_inicio) & (df_full[coluna_data].dt.date <= data_fim)].copy()
    if df_filtrado.empty: return f"Dados carregados com sucesso, mas não há registros para o período exato de {inicio_str} a {fim_str}."
    return f"Dados encontrados para '{nome_dataset}'. Total de registros: {len(df_filtrado)}.\nAmostra:\n{df_filtrado.head(3).to_string()}"

def analisador_de_dados_sob_demanda(query: str) -> str:
    """
    Carrega dados sob demanda para um período, executa uma análise complexa
    (filtros e operações como max/min/media) e retorna o resultado.
    A entrada deve ser: 'Nome do Dataset | AAAA-MM-DD | AAAA-MM-DD | Pergunta de Análise em JSON'
    """
    try:
        nome_dataset, inicio_str, fim_str, pergunta_analise_json = [x.strip() for x in query.split('|')]
        data_inicio = pd.to_datetime(inicio_str).date()
        data_fim = pd.to_datetime(fim_str).date()
    except (ValueError, IndexError):
        return "Entrada inválida. Use o formato 'Nome do Dataset | AAAA-MM-DD | AAAA-MM-DD | Pergunta de Análise em JSON'."

    print(f"\n[Ferramenta Otimizada On-Demand] Acionada para: {nome_dataset}, Período: {inicio_str} a {fim_str}")
    print(f"   - Pergunta de Análise: {pergunta_analise_json}")

    dataset_info = next((ds for cat in categorias_datasets.values() for ds in cat if ds['nome'] == nome_dataset), None)
    if not dataset_info: return f"Dataset '{nome_dataset}' não encontrado."

    prefixo = f"dataset/{dataset_info['valor']}/"
    file_structure = s3_file_structure_cache.get(prefixo, {})
    anos_no_periodo = range(data_inicio.year, data_fim.year + 1)
    arquivos_para_carregar = []
    for ano in anos_no_periodo:
        if ano in file_structure:
            dados_do_ano = file_structure[ano]
            arquivos_para_carregar.extend([v for k, v in dados_do_ano.items() if isinstance(k, int) or k == 'anual'])

    if not arquivos_para_carregar: return f"Nenhum arquivo de dados encontrado para o período."

    arquivos_para_carregar = sorted(list(set(arquivos_para_carregar)))
    df_full, status = carregar_dataframe_s3(arquivos_para_carregar)
    if df_full is None: return f"Falha ao carregar dados: {status}"

    coluna_data_principal = next((col for col, desc in descricoes_colunas.get(nome_dataset, {}).items() if 'data' in desc.lower()), None)
    if not coluna_data_principal: return "Não foi possível identificar a coluna de data principal no dicionário."

    df_full[coluna_data_principal] = pd.to_datetime(df_full[coluna_data_principal], errors='coerce')
    df_filtrado = df_full[(df_full[coluna_data_principal].dt.date >= data_inicio) & (df_full[coluna_data_principal].dt.date <= data_fim)].copy()

    if df_filtrado.empty: return f"Concluído. Nenhum dado encontrado para o período de {inicio_str} a {fim_str}."

    try:
        import json

        json_str = pergunta_analise_json.strip().strip('`').strip('json\n').strip()
        params = json.loads(json_str)

        filters = params.get("filters", [])
        columns_to_show = params.get("columns", [])
        operation = params.get("operation")

        for f in filters:
            col, op, val = f.get("column"), f.get("operator"), f.get("value")

            if 'dat_' in col or col.endswith('_data') or 'din_' in col:
                 df_filtrado.loc[:, col] = pd.to_datetime(df_filtrado[col], errors='coerce')

            if op == "year_equals":
                df_filtrado = df_filtrado[df_filtrado[col].dt.year == val]
            elif op == "equals":
                df_filtrado.loc[:, col] = pd.to_numeric(df_filtrado[col], errors='coerce')
                df_filtrado = df_filtrado[df_filtrado[col] == val]

        if df_filtrado.empty:
            return f"Concluído. Nenhum resultado encontrado após aplicar os filtros adicionais."

        if operation:
            op_type = operation.get("type")
            target_col = operation.get("column")

            if not target_col or target_col not in df_filtrado.columns:
                return f"Erro: A coluna '{target_col}' para a operação '{op_type}' não foi encontrada."

            df_filtrado.loc[:, target_col] = pd.to_numeric(df_filtrado[target_col], errors='coerce')

            if df_filtrado[target_col].dropna().empty:
                return f"Concluído. A coluna '{target_col}' não possui valores numéricos válidos."

            if op_type == "find_max":
                result_row = df_filtrado.loc[df_filtrado[target_col].idxmax()]
                df_result = result_row[columns_to_show] if columns_to_show else result_row
                return f"A análise foi concluída com sucesso. O valor máximo foi encontrado:\n{df_result.to_string()}"

            elif op_type == "find_min":
                result_row = df_filtrado.loc[df_filtrado[target_col].idxmin()]
                df_result = result_row[columns_to_show] if columns_to_show else result_row
                return f"A análise foi concluída com sucesso. O valor mínimo foi encontrado:\n{df_result.to_string()}"

            elif op_type == "calculate_mean":
                mean_value = df_filtrado[target_col].mean()
                return f"A análise foi concluída com sucesso. A média para '{target_col}' é: {mean_value:.2f}"

        df_result = df_filtrado[columns_to_show] if columns_to_show else df_filtrado
        return f"A análise foi concluída com sucesso. Encontrei os seguintes resultados:\n{df_result.to_string(index=False)}"

    except Exception as e:
        import traceback
        return f"Ocorreu um erro durante a análise on-demand: {e}\nTraceback: {traceback.format_exc()}"

def analisador_otimizado_pre_carregados(query: str) -> str:
    """
    Ferramenta otimizada que executa filtros e operações matemáticas (max, min, media, soma, contagem)
    diretamente no dataframe pré-carregado.
    """
    print(f"\n[Ferramenta Otimizada] Acionada com a query: {query}")
    try:
        import json
        json_str = query.strip().strip('`').strip('json\n').strip()
        params = json.loads(json_str)

        dataset_name = params.get("dataset")
        filters = params.get("filters", [])
        columns_to_show = params.get("columns", [])
        operation = params.get("operation")

        if not dataset_name:
            return "Erro: O nome do 'dataset' é obrigatório no JSON."

        df_filtered = dataframe_unificado_para_ia[dataframe_unificado_para_ia['nome_do_dataset'] == dataset_name].copy()

        if df_filtered.empty:
            return f"Nenhum dado encontrado para o dataset '{dataset_name}'."

        for f in filters:
            col, op, val = f.get("column"), f.get("operator"), f.get("value")

            if 'dat_' in col or col.endswith('_data') or 'din_' in col:
                 df_filtered.loc[:, col] = pd.to_datetime(df_filtered[col], errors='coerce')

            if op == "year_equals":
                df_filtered = df_filtered[df_filtered[col].dt.year == val]
            elif op == "equals":

                df_filtered.loc[:, col] = pd.to_numeric(df_filtered[col], errors='coerce')
                df_filtered = df_filtered[df_filtered[col] == val]

        if df_filtered.empty:
            return f"Concluído. Nenhum resultado encontrado para '{dataset_name}' após aplicar os filtros."

        if operation:
            op_type = operation.get("type")
            target_col = operation.get("column")

            if not target_col or target_col not in df_filtered.columns:
                return f"Erro: A coluna '{target_col}' para a operação '{op_type}' não foi encontrada."

            df_filtered.loc[:, target_col] = pd.to_numeric(df_filtered[target_col], errors='coerce')

            if df_filtered[target_col].dropna().empty:
                return f"Concluído. A coluna '{target_col}' não possui valores numéricos válidos para a operação."

            if op_type == "find_max" or op_type == "find_min":
                op_func = {'find_max': 'idxmax', 'find_min': 'idxmin'}[op_type]
                result_row = df_filtered.loc[df_filtered[target_col].agg(op_func)]

                cols_final = columns_to_show if columns_to_show else result_row.index.tolist()
                df_result = result_row[cols_final]

                op_text = "máximo" if op_type == "find_max" else "mínimo"
                return f"A análise foi concluída com sucesso. O valor {op_text} foi encontrado:\n{df_result.to_string()}"

            elif op_type == "calculate_mean":
                mean_value = df_filtered[target_col].mean()
                return f"A análise foi concluída com sucesso. A média para '{target_col}' é: {mean_value:.2f}"

            elif op_type == "calculate_sum":
                sum_value = df_filtered[target_col].sum()
                return f"A análise foi concluída com sucesso. A soma para '{target_col}' é: {sum_value:.2f}"

            elif op_type == "count":
                count_value = len(df_filtered)
                return f"A análise foi concluída com sucesso. A contagem de registros é: {count_value}"

        cols_final = columns_to_show if columns_to_show else df_filtered.columns.tolist()
        df_result = df_filtered[cols_final]

        return f"A análise foi concluída com sucesso. Encontrei os seguintes resultados:\n{df_result.to_string(index=False)}"

    except Exception as e:
        import traceback
        return f"Ocorreu um erro durante a análise otimizada: {e}\nTraceback: {traceback.format_exc()}"

def create_pandas_like_agent(llm, df, extra_tools=None):
    """
    Substituto moderno do create_pandas_dataframe_agent
    """

    @tool
    def visualizar_dataframe(linhas: int = 5) -> str:
        """Mostra as primeiras linhas do dataframe."""
        return df.head(linhas).to_string()

    @tool
    def descrever_dataframe() -> str:
        """Estatísticas descritivas do dataframe."""
        return df.describe(include="all").to_string()

    tools = [
        visualizar_dataframe,
        descrever_dataframe,
    ]

    if extra_tools:
        tools.extend(extra_tools)

    return create_react_agent(
        model=llm,
        tools=tools
    )

ferramenta_verificacao_disponibilidade = Tool(
    name="verificador_de_disponibilidade_de_dados",
    func=verificar_disponibilidade,
    description="Essencial para verificar a cobertura temporal (anos, meses) de um dataset. Use ANTES de tentar analisar os dados."
)

extrator_de_dados_por_data_tool = Tool(
    name="extrator_de_dados_por_data",
    func=extrator_de_dados_por_data,
    description="Use para obter um resumo SIMPLES ou amostra de dados de um dia ou período específico. Não use para análises complexas. Formato: 'Nome do Dataset | AAAA-MM-DD | AAAA-MM-DD'."
)

ferramenta_analise_sob_demanda = Tool(
    name="analisador_de_dados_sob_demanda",
    func=analisador_de_dados_sob_demanda,
    description="""Use para análises complexas (filtrar, max, min, media, etc) em datasets SOB DEMANDA.
    O input DEVE ser no formato de 4 partes separadas por '|':
    'Nome do Dataset | AAAA-MM-DD | AAAA-MM-DD | Análise em JSON'
    A quarta parte DEVE ser uma string JSON válida, começando com '{' e terminando com '}'.
    Exemplo de input: 'Dados Hidrológicos de Reservatórios - Diária | 2025-09-25 | 2025-09-25 | {"operation": {"type": "find_max", "column": "val_volumeutilcon"}}'"""
)

agente_pandas_precarregado = create_pandas_like_agent(
    llm,
    dataframe_unificado_para_ia,
    extra_tools=[
        ferramenta_verificacao_disponibilidade,
        extrator_de_dados_por_data_tool
    ]
)

ferramenta_analise_otimizada_memoria = Tool(
    name="analisador_otimizado_pre_carregados",
    func=analisador_otimizado_pre_carregados,
    description="""Ferramenta principal para análises RÁPIDAS em datasets pré-carregados. Ideal para filtrar e realizar operações matemáticas como 'maior', 'menor', 'média', 'soma' ou 'contagem'.
    A entrada DEVE ser EXCLUSIVAMENTE uma string JSON válida, começando com '{' e terminando com '}'.
    NÃO inclua marcadores de formatação (```) antes ou depois do JSON.

    Para executar uma operação, use a chave 'operation'. Tipos de operação disponíveis:
    - "find_max": Encontra a linha com o maior valor em uma coluna.
    - "find_min": Encontra a linha com o menor valor em uma coluna.
    - "calculate_mean": Calcula a média de uma coluna.
    - "calculate_sum": Calcula a soma de uma coluna.
    - "count": Conta o número de linhas após a filtragem.

    Sintaxe da operação: "operation": { "type": "TIPO_DA_OPERACAO", "column": "NOME_DA_COLUNA_ALVO" }

    Exemplo para encontrar a MÉDIA:
    {"dataset": "Carga de Energia Diária", "filters": [{"column": "nom_subsistema", "operator": "equals", "value": "SUDESTE"}], "operation": {"type": "calculate_mean", "column": "val_cargaenergiamwmed"}}
    """
)

# --- CORREÇÃO: Definição do Agente Pandas e da Função Wrapper ---

# 1. Recriamos as ferramentas internas do Pandas para que o Executor as reconheça
# (Isso é necessário porque elas foram definidas dentro de uma função no código anterior e o Executor precisa delas acessíveis)
@tool
def visualizar_dataframe(linhas: int = 5) -> str:
    """Mostra as primeiras linhas do dataframe para entender a estrutura."""
    # Garante que usa o dataframe global
    return dataframe_unificado_para_ia.head(linhas).to_string()

@tool
def descrever_dataframe() -> str:
    """Estatísticas descritivas (media, min, max) das colunas numéricas."""
    return dataframe_unificado_para_ia.describe(include="all").to_string()

# Lista de ferramentas que o sub-agente Pandas pode usar
tools_pandas_exec = [
    visualizar_dataframe,
    descrever_dataframe,
    ferramenta_verificacao_disponibilidade,
    extrator_de_dados_por_data_tool
]

# [cite_start]3. Definimos a função que estava faltando (O Wrapper) [cite: 119]
def analisador_generico_pre_carregados(query: str) -> str:
    try:
        resultado = agente_pandas_precarregado.invoke(
            {"messages": [("user", query)]}
        )
        return resultado["messages"][-1].content
    except Exception as e:
        return f"Erro ao processar análise genérica: {str(e)}"

# --- Fim da Correção ---

# Agora a ferramenta pode ser criada pois a função 'func' existe
ferramenta_analise_generica_memoria = Tool(
    name="analisador_generico_pre_carregados",
    func=analisador_generico_pre_carregados,
    description="Ferramenta flexível para análises abertas em datasets pré-carregados usando linguagem natural."
)

tools = [
    ferramenta_analise_otimizada_memoria,
    ferramenta_analise_generica_memoria,
    ferramenta_verificacao_disponibilidade,
    extrator_de_dados_por_data_tool,
    ferramenta_analise_sob_demanda,
]

bloco_dicionario = "### DICIONÁRIO DE DADOS (METADADOS DAS COLUNAS)\n\n"
for dataset, colunas in descricoes_colunas.items():
    bloco_dicionario += f"Para o dataset '{dataset}':\n"
    for nome_coluna, descricao in colunas.items():
        bloco_dicionario += f"  - A coluna `{nome_coluna}` significa: '{descricao}'.\n"
    bloco_dicionario += "\n"

datasets_preload = [d['nome'] for cat in categorias_datasets.values() for d in cat if d.get('estrategia_carga') == 'preload']
datasets_ondemand = [d['nome'] for cat in categorias_datasets.values() for d in cat if d.get('estrategia_carga') != 'preload']
bloco_mapa_dados = f"""### MAPA DE DADOS DISPONÍVEIS
1. Datasets Pré-Carregados (use `analisador_otimizado_pre_carregados` ou `analisador_generico_pre_carregados`): {", ".join(sorted(datasets_preload))}
2. Datasets Sob Demanda (use outras ferramentas): {", ".join(sorted(datasets_ondemand))}
"""
bloco_de_contexto = f"{bloco_dicionario}\n---\n{bloco_mapa_dados}"

base_template_string = """
Você é um agente de IA especialista em dados do Sistema Elétrico Brasileiro (ONS). Sua missão é usar o dicionário de dados e as ferramentas para entregar respostas precisas.

__BLOCO_DE_CONTEXTO__

Você tem acesso às seguintes ferramentas:
{tools}

Use o seguinte formato:
Thought: Seu raciocínio passo a passo. Comece SEMPRE consultando o dicionário de dados.
Action: O nome da ferramenta a ser usada (uma de [{tool_names}]).
Action Input: O input para a ferramenta.
Observation: O resultado da ferramenta.
... (Pode se repetir N vezes)
Thought: Eu agora sei a resposta final.
Final Answer: A resposta final e completa para a pergunta do usuário em português.
---
**REGRAS E FLUXO DE TRABALHO ATUALIZADO:**

**REGRA 0: CONSULTA OBRIGATÓRIA AO DICIONÁRIO E FORMATAÇÃO**
   - Antes de qualquer outra coisa, examine a pergunta e consulte o **DICIONÁRIO DE DADOS** acima para identificar as colunas corretas.
   - No seu raciocínio (Thought), declare explicitamente quais colunas você vai usar com base no dicionário.
   - **NÃO presuma nomes de colunas.** Verifique no dicionário primeiro.
   - **Após receber uma Observation de uma ferramenta, seu primeiro passo é avaliar se ela já contém a resposta final.** Se a `Observation` contém a resposta para a pergunta do usuário (seja uma lista, um número ou uma frase conclusiva), seu próximo `Thought` deve ser "Eu agora sei a resposta final." e

**REGRA 1: ESCOLHA DA ESTRATÉGIA (CONCEITUAL VS. DADOS)**
   - **Se a pergunta for conceitual, conversacional ou geral** (ex: "olá", "você está pronto?", "obrigado"): Seu raciocínio (Thought) deve ser breve, e você **DEVE** fornecer a resposta diretamente no campo "Final Answer:". **NÃO responda com texto simples.**
     Exemplo de fluxo para uma pergunta como "Olá, tudo bem?":
     Thought: O usuário está fazendo uma saudação. Devo responder cordialmente e me colocar à disposição.
     Final Answer: Olá! Tudo bem. Estou pronto para ajudar com seus dados. Qual é a sua pergunta?
   - **Se a pergunta exigir dados:** Identifique o dataset necessário usando o **MAPA DE DADOS** e prossiga para a Regra 2.

**REGRA 2: ESCOLHA DA FERRAMENTA DE ANÁLISE**
   - Se a pergunta for uma **FILTRAGEM DIRETA ou OPERAÇÃO MATEMÁTICA** (max, min, media, etc.):
     - Monte a análise como uma **string JSON** (com 'filters', 'operation', etc.).
     - Se o dataset for **Pré-Carregado**: Use a ferramenta RÁPIDA `analisador_otimizado_pre_carregados` com o JSON como input.
     - Se o dataset for **Sob Demanda**: Use a ferramenta `analisador_de_dados_sob_demanda` no formato 'Nome do Dataset | Data Início | Data Fim | String JSON'.

   - Se a pergunta for **ABERTA ou MUITO COMPLEXA** e o dataset for **Pré-Carregado**: Use a ferramenta FLEXÍVEL `analisador_generico_pre_carregados` com a pergunta em linguagem natural.
---
Pergunta: {input}
Thought:{agent_scratchpad}
"""

prompt_template = PromptTemplate(
    input_variables=["input", "agent_scratchpad", "tools", "tool_names"],
    template=base_template_string.replace(
        "__BLOCO_DE_CONTEXTO__", bloco_de_contexto
    )
)

agent_principal = create_react_agent(
    model=llm,
    tools=tools,
    prompt=prompt_template
)
print("\n✅ Agente de IA com dicionário de dados (versão final) está pronto!")

In [ ]:
# --- Bloco 2: Interface Gradio

def atualizar_datasets_dropdown(categoria):
    if not categoria:
        return gr.Dropdown(choices=[], value=None, interactive=False), "Selecione uma categoria.", gr.Accordion(visible=False), gr.DataFrame(visible=False)
    nomes_datasets = [ds['nome'] for ds in categorias_datasets.get(categoria, [])]
    return gr.Dropdown(choices=sorted(nomes_datasets), value=None, interactive=True), "Selecione um dataset.", gr.Accordion(visible=False), gr.DataFrame(visible=False)

def get_s3_columns_from_any_file(file_structure):
    if not file_structure: return ["[ERRO] Estrutura de arquivos vazia."]
    try:
        first_key = next(iter(file_structure))
        if isinstance(first_key, int):
            first_file_key = next(iter(file_structure[first_key].values()))
        else:
            first_file_key = file_structure[first_key][0]
        temp_filename = f"temp_cols_{os.path.basename(first_file_key).replace('/', '_')}"
        s3_client.download_file(bucket_name, first_file_key, temp_filename)
        try:
            df = pd.read_csv(temp_filename, sep=';', decimal=',', nrows=5, encoding='utf-8')
        except Exception:
            df = pd.read_csv(temp_filename, sep=',', decimal='.', nrows=5, encoding='utf-8')
        if os.path.exists(temp_filename): os.remove(temp_filename)
        return df.columns.tolist()
    except Exception as e:
        if 'temp_filename' in locals() and os.path.exists(temp_filename): os.remove(temp_filename)
        return [f"[ERRO] Falha ao ler colunas: {e}"]

def analisar_dataset_e_preparar_filtros(categoria, nome_dataset):
    if not nome_dataset:
        return "Aguardando seleção...", gr.Accordion(visible=False), gr.Dropdown(), gr.Dropdown(), gr.CheckboxGroup(), gr.DataFrame()
    dataset_info = next((ds for ds in categorias_datasets[categoria] if ds['nome'] == nome_dataset), None)
    prefixo = f"dataset/{dataset_info['valor']}/"
    file_structure = s3_file_structure_cache.get(prefixo, {})
    anos_disponiveis = sorted([k for k in file_structure.keys() if isinstance(k, int)], reverse=True)
    if not file_structure:
        return f"Nenhum arquivo encontrado para '{nome_dataset}'.", gr.Accordion(visible=False), gr.Dropdown(), gr.Dropdown(), gr.CheckboxGroup(), gr.DataFrame()
    colunas = get_s3_columns_from_any_file(file_structure)
    descricoes = descricoes_colunas.get(nome_dataset, {})
    opcoes_colunas = [(descricoes.get(col, col), col) for col in colunas]
    ano_dropdown_update = gr.Dropdown(choices=anos_disponiveis, value=anos_disponiveis[0] if anos_disponiveis else None, label="Ano", visible=bool(anos_disponiveis), interactive=True)
    mes_dropdown_update = gr.Dropdown(label="Mês (Opcional)", choices=[("Todos", None)] + [(f"{i:02d}", i) for i in range(1, 13)], visible=bool(anos_disponiveis), interactive=True, value=None)
    return ("Filtros prontos.", gr.Accordion(visible=True, open=True), ano_dropdown_update, mes_dropdown_update,
            gr.CheckboxGroup(choices=opcoes_colunas, value=[c[1] for c in opcoes_colunas], visible=True),
            gr.DataFrame(visible=False))

def carregar_dados_filtrados(categoria, nome_dataset, ano, mes, colunas_selecionadas, num_linhas):
    if not nome_dataset:
        return "Selecione um dataset.", gr.DataFrame(visible=False), gr.Slider()
    dataset_info = next((ds for ds in categorias_datasets[categoria] if ds['nome'] == nome_dataset), None)
    df, status, display_source = None, "", ""
    if dataset_info['estrategia_carga'] == 'preload':
        try:
            df = next(d.copy() for d in lista_de_dataframes_para_ia if d['nome_do_dataset'].iloc[0] == nome_dataset)
            display_source = "memória (pré-carregado)"
        except (NameError, StopIteration):
            status = f"Erro: '{nome_dataset}' não foi encontrado no pré-carregamento."
    if df is None:
        prefixo = f"dataset/{dataset_info['valor']}/"
        file_structure = s3_file_structure_cache.get(prefixo, {})
        arquivos_para_carregar = []
        if ano:
            ano_int = int(ano)
            if ano_int in file_structure:
                dados_do_ano = file_structure[ano_int]
                mes_int = int(mes) if mes else None
                if mes_int and mes_int in dados_do_ano:
                    arquivos_para_carregar = [dados_do_ano[mes_int]]
                    display_source = f"S3 (arquivo de {ano}-{mes_int:02d})"
                else:
                    arquivos_para_carregar.extend([v for k, v in dados_do_ano.items() if isinstance(k, int) or k == 'anual'])
                    display_source = f"S3 ({len(arquivos_para_carregar)} arquivo(s) de {ano})"
        if not arquivos_para_carregar and 'atemporal' in file_structure:
            arquivos_para_carregar = file_structure['atemporal']
            display_source = "S3 (arquivo atemporal)"
        if arquivos_para_carregar: df, status = carregar_dataframe_s3(arquivos_para_carregar)
        else: status = "Nenhum arquivo encontrado para a seleção."
    if df is None:
        return f"Falha no carregamento: {status}", gr.DataFrame(visible=False), gr.Slider()

    col_data = next((col for col in df.columns if 'dat_' in col or 'din_' in col), None)
    if ano and col_data:
        df[col_data] = pd.to_datetime(df[col_data], errors='coerce')
        df.dropna(subset=[col_data], inplace=True)
        df = df[df[col_data].dt.year == int(ano)]
    if mes and col_data:
        df = df[df[col_data].dt.month == int(mes)]

    colunas_validas = [col for col in colunas_selecionadas if col in df.columns]
    df_filtrado = df[colunas_validas] if colunas_validas else df
    total_linhas = len(df_filtrado)
    if total_linhas == 0:
        return f"Nenhum registro encontrado para os filtros. Fonte: {display_source}.", gr.DataFrame(visible=False), gr.Slider()
    slider_update = gr.Slider(minimum=10, maximum=max(10, total_linhas), value=min(num_linhas, max(10, total_linhas)), step=10)
    status_final = f"Exibindo {min(num_linhas, total_linhas)} de {total_linhas} linhas. Fonte: {display_source}."
    return status_final, gr.DataFrame(value=df_filtrado.head(num_linhas), visible=True), slider_update

def chat_with_agent_and_update_ui(message, history):
    history = history or []
    if not agent:
        history.append((message, "Desculpe, o agente de IA não foi inicializado."))
        return history, gr.DataFrame(visible=False)
    print(f"Recebida nova pergunta para a IA: {message}")
    try:
        response_dict = agent.invoke({"input": message})
        full_answer = response_dict.get('output', "Não consegui encontrar uma resposta.")
    except Exception as e:
        print(f"Ocorreu um erro na execução do agente: {e}")
        history.append((message, f"Desculpe, ocorreu um erro: {e}"))
        return history, gr.DataFrame(visible=False)

    text_answer = full_answer
    final_df = None
    csv_pattern = r"```csv\n(.*?)\n```"
    match = re.search(csv_pattern, full_answer, re.DOTALL)
    if match:
        csv_str = match.group(1)
        text_answer = full_answer.replace(match.group(0), "\n\n(Veja a tabela de dados abaixo)").strip()
        try: final_df = pd.read_csv(StringIO(csv_str))
        except Exception as err: text_answer += f"\n(Erro ao ler tabela: {err})"

    history.append((message, text_answer))
    return history, gr.DataFrame(value=final_df, visible=(final_df is not None))

with gr.Blocks(theme=gr.themes.Default(primary_hue="blue", secondary_hue="blue")) as demo_final:
    gr.Markdown("# 📊 Ferramenta de Análise de Dados do ONS (Versão Integrada)")
    with gr.Tabs():
        with gr.TabItem("Explorador Visual com Filtros"):
            gr.Markdown("Selecione uma categoria e um dataset para analisar, filtrar e visualizar.")
            with gr.Row():
                dropdown_categoria = gr.Dropdown(label="1. Categoria", choices=sorted(list(categorias_datasets.keys())))
                dropdown_dataset = gr.Dropdown(label="2. Dataset", interactive=False)
            txt_status = gr.Textbox(label="Status", interactive=False, value="Aguardando seleção...")
            with gr.Accordion("Filtros e Visualização", open=False, visible=False) as accordion_filtros:
                with gr.Row():
                    dropdown_ano = gr.Dropdown(label="Ano", visible=True)
                    dropdown_mes = gr.Dropdown(label="Mês (Opcional)", visible=True)
                chkgroup_colunas = gr.CheckboxGroup(label="Colunas do Dataset", visible=True)
                slider_linhas = gr.Slider(label="Número de linhas para exibir", minimum=10, maximum=500, value=50, step=10)
                btn_carregar = gr.Button("Carregar e Visualizar Dados", variant="primary")
            output_dataframe = gr.DataFrame(label="Pré-visualização dos Dados", visible=False, interactive=False)

        with gr.TabItem("Chatbot com IA"):
            gr.Markdown("Faça perguntas em português sobre os dados do sistema elétrico.")
            chatbot_display = gr.Chatbot(height=500, label="Conversa")
            with gr.Row():
                msg_input = gr.Textbox(scale=4, show_label=False, placeholder="Ex: qual a soma da potência efetiva por estado no dataset de Capacidade Instalada?", container=False)
                submit_btn = gr.Button("Enviar")
            chatbot_dataframe_output = gr.DataFrame(label="Dados da Resposta", visible=False, interactive=False)

    outputs_analise = [txt_status, accordion_filtros, dropdown_ano, dropdown_mes, chkgroup_colunas, output_dataframe]
    inputs_carregar = [dropdown_categoria, dropdown_dataset, dropdown_ano, dropdown_mes, chkgroup_colunas, slider_linhas]

    dropdown_categoria.change(fn=atualizar_datasets_dropdown, inputs=dropdown_categoria, outputs=[dropdown_dataset, txt_status, accordion_filtros, output_dataframe])
    dropdown_dataset.change(fn=analisar_dataset_e_preparar_filtros, inputs=[dropdown_categoria, dropdown_dataset], outputs=outputs_analise)
    btn_carregar.click(fn=carregar_dados_filtrados, inputs=inputs_carregar, outputs=[txt_status, output_dataframe, slider_linhas])

    submit_btn.click(fn=chat_with_agent_and_update_ui, inputs=[msg_input, chatbot_display], outputs=[chatbot_display, chatbot_dataframe_output]).then(lambda: gr.Textbox(value=""), outputs=[msg_input])
    msg_input.submit(fn=chat_with_agent_and_update_ui, inputs=[msg_input, chatbot_display], outputs=[chatbot_display, chatbot_dataframe_output]).then(lambda: gr.Textbox(value=""), outputs=[msg_input])


print("\nIniciando a interface Gradio da versão final integrada...")
demo_final.launch(debug=True, show_error=True)


/tmp/ipython-input-470/2922866891.py:125: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Default(primary_hue="blue", secondary_hue="blue")) as demo_final:



Iniciando a interface Gradio da versão final integrada...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f0290388832c145dcc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
